# F2-M04: Análisis Univariante Numérico

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Análisis univariante de todas las variables numéricas: clasificación por
naturaleza (continua/discreta), estadísticas descriptivas, detección de
outliers (IQR), forma de distribuciones (asimetría, curtosis), histogramas
con KDE, QQ-plots de normalidad, violinplots por curso y
stripplots de outliers.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`
- Librerías: `scipy.stats`, `plotly`

## Genera
- `docs/html/fase2/m04_univariante_num.html`
- `docs/html/fase2/graficos/m04_clasificacion.html`
- `docs/html/fase2/graficos/m04_boxplots.html` (sin outliers)
- `docs/html/fase2/graficos/m04_boxplots_full.html` (con outliers, modal)
- `docs/html/fase2/graficos/m04_forma.html`
- `docs/html/fase2/graficos/m04_histogramas.html`
- `docs/html/fase2/graficos/m04_qqplot.html`
- `docs/html/fase2/graficos/m04_violinplots.html` *(NUEVO)*
- `docs/html/fase2/graficos/m04_stripplot.html` *(NUEVO)*

## Flujo
```
... → M03 Nulos → **M04 Univariante Num** → M05 Univariante Cat → ...
```

## Siguiente
`f2_m05_univariante_cat.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto, Plotly y scipy
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: IDENTIFICAR Y CLASIFICAR VARIABLES NUMÉRICAS
# ============================================================================
# Excluye IDs, fechas y categóricas codificadas como número.
# Clasifica por cardinalidad:
#   - Discreta (≤5): pocas categorías
#   - Discreta (6-20): cardinalidad media
#   - Continua (>20): variable continua
# ============================================================================

print('=' * 60)
print('ANÁLISIS UNIVARIANTE NUMÉRICO')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# Variables excluidas del análisis numérico
# - IDs: per_id_ficticio, exp_tit_id
# - Fechas: curso_aca, curso_aca_id, curso_aca_ini, curso_aca_fin
# - Binaria codificada: sexo (0/1, sin gradación)
# - orden_preferencia: variable ordinal numérica (rango 1-20) excluida porque
#   el 12.6% de registros tiene NaN estructural — alumnos sin preinscripción
#   (traslados, mayores de 25, accesos directos). Se analiza por separado en F2-M04b.
excluir = ['per_id_ficticio', 'exp_tit_id', 'curso_aca', 'curso_aca_id',
           'curso_aca_ini', 'curso_aca_fin', 'sexo', 'orden_preferencia']

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in excluir]

# Clasificar por naturaleza y cardinalidad
clasificacion = []
for col in num_cols:
    n_unicos = df[col].nunique()
    if n_unicos <= 5:
        tipo = 'Discreta (≤5)'
        categoria = 'discreta'
    elif n_unicos <= 20:
        tipo = 'Discreta (6-20)'
        categoria = 'discreta'
    else:
        tipo = 'Continua (>20)'
        categoria = 'continua'
    clasificacion.append({'Variable': col, 'Tipo': tipo, 'Categoria': categoria, 'Unicos': n_unicos})

df_clasif = pd.DataFrame(clasificacion)
continuas = df_clasif[df_clasif['Categoria'] == 'continua']['Variable'].tolist()
discretas = df_clasif[df_clasif['Categoria'] == 'discreta']['Variable'].tolist()

# Variables con >5 valores únicos (candidatas a histograma y QQ-plot)
vars_hist = df_clasif[df_clasif['Unicos'] > 5]['Variable'].tolist()

print(f'📊 Variables numéricas: {len(num_cols)}')
print(f'   - Continuas (>20): {len(continuas)}')
print(f'   - Discretas (≤20): {len(discretas)}')
print(f'   - Para histogramas (>5 únicos): {len(vars_hist)}')

ANÁLISIS UNIVARIANTE NUMÉRICO


📊 Variables numéricas: 10
   - Continuas (>20): 8
   - Discretas (≤20): 2
   - Para histogramas (>5 únicos): 9


In [3]:
# ============================================================================
# CELDA 3: CALCULAR ESTADÍSTICAS DESCRIPTIVAS Y OUTLIERS
# ============================================================================
# Para cada variable numérica calcula:
# - Descriptivos: media, std, min, Q1, mediana, Q3, max
# - Forma: asimetría (skew) y curtosis (sin outliers)
# - Outliers: método IQR (1.5×IQR)
# - Valores extremos: ratio max/mediana > 5
# ============================================================================

print('\n📊 Calculando estadísticas...')

stats_list = []
outliers_info = []

for col in num_cols:
    data = df[col].dropna()
    if len(data) == 0:
        continue

    # Cuartiles y rango intercuartílico
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Datos sin outliers (para forma)
    data_clean = data[(data >= lower) & (data <= upper)]
    n_outliers = len(data) - len(data_clean)

    mediana = data.median()
    skewness = stats.skew(data_clean) if len(data_clean) > 0 else 0
    kurt = stats.kurtosis(data_clean) if len(data_clean) > 0 else 0

    stats_list.append({
        'Variable': col,
        'Count': len(data),
        'Mean': data.mean(),
        'Std': data.std(),
        'Min': data.min(),
        '25%': q1,
        '50%': mediana,
        '75%': q3,
        'Max': data.max(),
        'Missing': df[col].isnull().sum(),
        'Missing%': df[col].isnull().mean() * 100,
        'Skew': skewness,
        'Kurtosis': kurt,
        'Outliers': n_outliers
    })

    # Detectar valores extremos (ratio max/mediana > 5)
    if data.max() > mediana * 5 and mediana > 0:
        outliers_info.append({
            'Variable': col, 'Tipo': 'MAX',
            'Valor Extremo': data.max(), 'Mediana': mediana,
            'Ratio': data.max() / mediana, 'N Outliers': (data > upper).sum()
        })
    if data.min() == 0 and mediana > 0 and (data < lower).sum() > 0:
        outliers_info.append({
            'Variable': col, 'Tipo': 'MIN',
            'Valor Extremo': data.min(), 'Mediana': mediana,
            'Ratio': 0, 'N Outliers': (data < lower).sum()
        })

df_stats = pd.DataFrame(stats_list)
df_outliers = pd.DataFrame(outliers_info).sort_values('Ratio', ascending=False) if outliers_info else pd.DataFrame()
n_con_outliers = (df_stats['Outliers'] > 0).sum()

print(f'✅ {len(df_stats)} variables | ⚠️ {n_con_outliers} con outliers')


📊 Calculando estadísticas...
✅ 10 variables | ⚠️ 7 con outliers


### 📌 Nota metodológica: `cred_titulacion` — Skew=NaN y Kurtosis=NaN

La variable `cred_titulacion` (créditos totales de la titulación) presenta **Skew=NaN**
y **Kurtosis=NaN** en la tabla de estadísticas descriptivas. Esto no es un error.

**Causa:** `cred_titulacion` tiene varianza prácticamente nula. La gran mayoría de
alumnos pertenecen a titulaciones de **240 créditos** (grados de 4 años). Un grupo
reducido pertenece a titulaciones de **300 o 384 créditos** (grados de 5 años como
Medicina o Arquitectura). Al eliminar outliers por el método IQR, los pocos valores
distintos de 240 quedan fuera del rango — los datos restantes son constantes.

**Los estadísticos de forma no están definidos para datos constantes** —
`scipy.stats.skew()` devuelve NaN de forma correcta y esperada, no por un bug.

**Interpretación:** `cred_titulacion` no es una variable continua real — es una
variable discreta cuyos valores están determinados por el plan de estudios de cada
titulación. Sus valores posibles son fijos (240, 300, 384). No se incluye en el
análisis de forma ni en los histogramas por esta razón.


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO — CLASIFICACIÓN POR NATURALEZA (DONUT)
# ============================================================================
# Donut mostrando proporción de continuas vs discretas.
# ============================================================================

print('=' * 60)
print('GENERANDO GRÁFICOS')
print('=' * 60)

tipo_counts = df_clasif['Tipo'].value_counts()

colores_tipo = {
    'Continua (>20)': '#2B6CB0',
    'Discreta (6-20)': '#48BB78',
    'Discreta (≤5)': '#9AE6B4'
}
colores = [colores_tipo.get(t, '#718096') for t in tipo_counts.index]

fig_clasif = go.Figure(go.Pie(
    labels=tipo_counts.index,
    values=tipo_counts.values,
    hole=0.4,
    marker_colors=colores,
    textinfo='label+value+percent',
    textposition='outside'
))
fig_clasif.update_layout(
    title='📊 Clasificación por Naturaleza',
    height=400,
    margin=dict(t=60, b=40, l=40, r=40),
    showlegend=False
)
fig_clasif.write_html(RUTA_GRAFICOS / 'm04_clasificacion.html', include_plotlyjs='cdn')
print('✅ Gráfico clasificación')

GENERANDO GRÁFICOS


✅ Gráfico clasificación


In [5]:
# ============================================================================
# CELDA 5: BOXPLOTS SIN/CON OUTLIERS
# ============================================================================
# Genera dos versiones de boxplots normalizados (z-score):
# 1. Sin outliers extremos → vista principal
# 2. Con outliers → accesible via modal popup
# ============================================================================

colores_box = px.colors.qualitative.Set2

# --- Boxplots SIN outliers ---
fig_box = go.Figure()
for i, col in enumerate(vars_hist):
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    data_vis = data[(data >= q1 - 1.5*iqr) & (data <= q3 + 1.5*iqr)]

    if len(data_vis) > 0 and data_vis.std() > 0:
        data_norm = (data_vis - data_vis.mean()) / data_vis.std()
        fig_box.add_trace(go.Box(
            y=data_norm, name=col,
            marker_color=colores_box[i % len(colores_box)],
            boxmean='sd'
        ))

fig_box.update_layout(
    title='📦 Distribuciones (sin outliers extremos)',
    yaxis_title='Valor',
    height=420,
    margin=dict(t=60, b=100, l=60, r=40),
    showlegend=False,
    xaxis_tickangle=45
)
fig_box.write_html(RUTA_GRAFICOS / 'm04_boxplots.html', include_plotlyjs='cdn')

# --- Boxplots CON outliers (para modal, muestra de 5000 para rendimiento) ---
fig_box_full = go.Figure()
for i, col in enumerate(vars_hist):
    data = df[col].dropna()
    if len(data) > 5000:
        data = data.sample(5000, random_state=42)
    if len(data) > 0 and data.std() > 0:
        data_norm = (data - data.mean()) / data.std()
        fig_box_full.add_trace(go.Box(
            y=data_norm, name=col,
            marker_color=colores_box[i % len(colores_box)],
            boxmean='sd'
        ))

fig_box_full.update_layout(
    title='📦 Distribuciones (con outliers)',
    yaxis_title='Valor', height=450,
    margin=dict(t=60, b=100, l=60, r=40),
    showlegend=False, xaxis_tickangle=45
)
fig_box_full.write_html(RUTA_GRAFICOS / 'm04_boxplots_full.html', include_plotlyjs='cdn')
print('✅ Boxplots generados')

✅ Boxplots generados


In [6]:
# ============================================================================
# CELDA 6: ASIMETRÍA Y CURTOSIS
# ============================================================================
# Gráfico de barras lado a lado mostrando skewness y kurtosis.
# Amarillo = positivo, Azul = negativo.
# Calculados sobre datos sin outliers.
# ============================================================================

df_forma = df_stats[['Variable', 'Skew', 'Kurtosis']].copy()

fig_forma = make_subplots(rows=1, cols=2, subplot_titles=['Asimetría (Skewness)', 'Curtosis'])

colores_skew = ['#D69E2E' if s > 0 else '#3182CE' for s in df_forma['Skew']]
fig_forma.add_trace(go.Bar(
    x=df_forma['Variable'], y=df_forma['Skew'],
    marker_color=colores_skew, name='Asimetría'
), row=1, col=1)

colores_kurt = ['#D69E2E' if k > 0 else '#3182CE' for k in df_forma['Kurtosis']]
fig_forma.add_trace(go.Bar(
    x=df_forma['Variable'], y=df_forma['Kurtosis'],
    marker_color=colores_kurt, name='Curtosis'
), row=1, col=2)

fig_forma.update_layout(
    title='📐 Forma de las Distribuciones (sin outliers)',
    height=400,
    margin=dict(t=80, b=100, l=60, r=40),
    showlegend=False
)
fig_forma.update_xaxes(tickangle=45)
fig_forma.write_html(RUTA_GRAFICOS / 'm04_forma.html', include_plotlyjs='cdn')
print('✅ Gráfico forma (asimetría + curtosis)')

✅ Gráfico forma (asimetría + curtosis)


In [7]:
# ============================================================================
# CELDA 7: HISTOGRAMAS CON KDE
# ============================================================================
# Grid de histogramas (3 columnas) para variables con >5 únicos.
# Cada subplot incluye:
# - Histograma normalizado (densidad)
# - Curva KDE (gaussian_kde)
# - Línea roja: media | Línea verde punteada: mediana
# Datos filtrados sin outliers IQR.
# ============================================================================

n_vars = len(vars_hist)
n_cols_grid = 3
n_rows_grid = (n_vars + n_cols_grid - 1) // n_cols_grid

colores_hist = px.colors.qualitative.Set2

fig_hist = make_subplots(
    rows=n_rows_grid, cols=n_cols_grid,
    subplot_titles=vars_hist
)

for i, col in enumerate(vars_hist):
    row = i // n_cols_grid + 1
    col_idx = i % n_cols_grid + 1

    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    data_vis = data[(data >= q1 - 1.5*iqr) & (data <= q3 + 1.5*iqr)]

    if len(data_vis) > 0:
        # Histograma normalizado
        fig_hist.add_trace(go.Histogram(
            x=data_vis, nbinsx=30,
            marker_color=colores_hist[i % len(colores_hist)],
            opacity=0.7,
            histnorm='probability density',
            name=col, showlegend=False
        ), row=row, col=col_idx)

        # Curva KDE superpuesta
        try:
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(data_vis)
            x_range = np.linspace(data_vis.min(), data_vis.max(), 100)
            fig_hist.add_trace(go.Scatter(
                x=x_range, y=kde(x_range),
                mode='lines', line=dict(color='black', width=2),
                name='KDE', showlegend=False
            ), row=row, col=col_idx)
        except Exception:
            pass

        # Líneas de referencia: media (roja) y mediana (verde)
        fig_hist.add_vline(x=data_vis.mean(), line_dash='solid', line_color='red', row=row, col=col_idx)
        fig_hist.add_vline(x=data_vis.median(), line_dash='dash', line_color='green', row=row, col=col_idx)

fig_hist.update_layout(
    title=f'📊 Distribución de Frecuencias ({n_vars} variables con >5 valores únicos)',
    height=300 * n_rows_grid,
    margin=dict(t=80, b=60, l=60, r=40),
    showlegend=False
)
fig_hist.write_html(RUTA_GRAFICOS / 'm04_histogramas.html', include_plotlyjs='cdn')
print(f'✅ Histogramas con KDE ({n_vars} variables)')

✅ Histogramas con KDE (9 variables)


In [8]:
# ============================================================================
# CELDA 8: QQ-PLOTS DE NORMALIDAD
# ============================================================================
# Grid de QQ-plots (3 columnas) comparando cuantiles observados
# vs teóricos de una distribución normal.
# Línea roja punteada = referencia de normalidad perfecta.
# Datos filtrados sin outliers IQR.
# ============================================================================

fig_qq = make_subplots(
    rows=n_rows_grid, cols=n_cols_grid,
    subplot_titles=vars_hist
)

for i, col in enumerate(vars_hist):
    row = i // n_cols_grid + 1
    col_idx = i % n_cols_grid + 1

    data = df[col].dropna().sort_values()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    data_vis = data[(data >= q1 - 1.5*iqr) & (data <= q3 + 1.5*iqr)]

    if len(data_vis) > 10:
        n = len(data_vis)
        theoretical_q = stats.norm.ppf(np.linspace(0.01, 0.99, n))
        sample_q = np.sort(data_vis.values)

        # Puntos observados
        fig_qq.add_trace(go.Scatter(
            x=theoretical_q, y=sample_q,
            mode='markers', marker=dict(color='#3182CE', size=4),
            name=col, showlegend=False
        ), row=row, col=col_idx)

        # Línea de referencia (ajuste lineal)
        slope, intercept = np.polyfit(theoretical_q, sample_q, 1)
        line_y = slope * theoretical_q + intercept
        fig_qq.add_trace(go.Scatter(
            x=theoretical_q, y=line_y,
            mode='lines', line=dict(color='red', dash='dash'),
            name='Referencia', showlegend=False
        ), row=row, col=col_idx)

fig_qq.update_layout(
    title='📈 QQ-Plot: ¿Distribución Normal? (variables con >5 valores únicos)',
    height=300 * n_rows_grid,
    margin=dict(t=80, b=60, l=60, r=40),
    showlegend=False
)
fig_qq.write_html(RUTA_GRAFICOS / 'm04_qqplot.html', include_plotlyjs='cdn')
print(f'✅ QQ-Plots ({n_vars} variables)')

✅ QQ-Plots (9 variables)


In [9]:
# ============================================================================
# CELDA 9: GRÁFICO NUEVO — VIOLINPLOTS DE VARIABLES CLAVE POR CURSO
# ============================================================================
# Idea del profesor: violinplot muestra la FORMA de la distribución
# (no solo caja), revelando bimodalidad, asimetrías, etc.
# Se seleccionan las variables numéricas más relevantes para abandono
# y se muestran agrupadas por curso académico.
# ============================================================================

# Variables clave para violinplots (continuas con más de 20 únicos)
vars_violin = [c for c in continuas if c in df.columns][:6]  # Máx 6

# Detectar columna de curso
curso_col = None
for col in ['curso_aca', 'curso_aca_id']:
    if col in df.columns:
        curso_col = col
        break

if vars_violin and curso_col:
    from plotly.subplots import make_subplots

    n_vars_v = len(vars_violin)
    n_cols_v = 2
    n_rows_v = (n_vars_v + n_cols_v - 1) // n_cols_v

    fig_violin = make_subplots(
        rows=n_rows_v, cols=n_cols_v,
        subplot_titles=vars_violin
    )

    cursos_unicos = sorted(df[curso_col].dropna().unique())
    # Paleta de colores para cursos
    colores_curso = px.colors.qualitative.Set2[:len(cursos_unicos)]

    for i, var in enumerate(vars_violin):
        row = i // n_cols_v + 1
        col_idx = i % n_cols_v + 1

        for j, curso in enumerate(cursos_unicos):
            data_curso = df[df[curso_col] == curso][var].dropna()
            if len(data_curso) > 0:
                fig_violin.add_trace(go.Violin(
                    y=data_curso,
                    name=str(curso),
                    marker_color=colores_curso[j % len(colores_curso)],
                    box_visible=True,       # Caja dentro del violín
                    meanline_visible=True,  # Línea de media
                    showlegend=(i == 0),    # Leyenda solo en el primero
                    legendgroup=str(curso),
                    scalemode='width',
                    points=False            # Sin puntos (los ponemos en stripplot)
                ), row=row, col=col_idx)

    fig_violin.update_layout(
        title='🎻 Violinplots: Distribuciones por Curso Académico',
        height=350 * n_rows_v,
        margin=dict(t=60, b=100, l=60, r=40),
        showlegend=True,
        legend=dict(orientation='h', yanchor='top', y=-0.05, xanchor='center', x=0.5,
                    title='Curso')
    )
    fig_violin.write_html(RUTA_GRAFICOS / 'm04_violinplots.html', include_plotlyjs='cdn')
    print(f'✅ Violinplots: {n_vars_v} variables × {len(cursos_unicos)} cursos')
else:
    print('⚠️ No se generan violinplots (sin variables continuas o sin curso)')


✅ Violinplots: 6 variables × 11 cursos


In [10]:
# ============================================================================
# CELDA 10: GRÁFICO NUEVO — STRIPPLOT DE OUTLIERS
# ============================================================================
# Idea del profesor: puntos individuales superpuestos muestran los
# outliers reales (no solo la marca del boxplot).
# Se seleccionan las variables con más outliers y se muestran
# como puntos jittered con la mediana y los límites IQR.
# ============================================================================

# Variables con outliers (top 6 por número de outliers)
df_con_outliers = df_stats[df_stats['Outliers'] > 0].nlargest(6, 'Outliers')
vars_strip = df_con_outliers['Variable'].tolist()

if vars_strip:
    from plotly.subplots import make_subplots

    n_vars_s = len(vars_strip)
    n_cols_s = 2
    n_rows_s = (n_vars_s + n_cols_s - 1) // n_cols_s

    fig_strip = make_subplots(
        rows=n_rows_s, cols=n_cols_s,
        subplot_titles=[f'{v} ({int(df_con_outliers[df_con_outliers["Variable"]==v]["Outliers"].values[0]):,} outliers)'
                        for v in vars_strip]
    )

    for i, var in enumerate(vars_strip):
        row = i // n_cols_s + 1
        col_idx = i % n_cols_s + 1

        data = df[var].dropna()
        q1, q3 = data.quantile(0.25), data.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        # Separar normales y outliers
        normales = data[(data >= lower) & (data <= upper)]
        outliers_vals = data[(data < lower) | (data > upper)]

        # Muestra aleatoria de normales (máx 300 para rendimiento)
        if len(normales) > 300:
            normales_muestra = normales.sample(300, random_state=42)
        else:
            normales_muestra = normales

        # Puntos normales (gris, jittered)
        fig_strip.add_trace(go.Box(
            y=normales_muestra,
            name=var,
            marker=dict(color='#A0AEC0', size=3, opacity=0.4),
            boxpoints='all',
            jitter=0.6,
            pointpos=0,
            line=dict(color='#3182CE', width=2),
            fillcolor='rgba(49,130,206,0.1)',
            showlegend=False
        ), row=row, col=col_idx)

        # Puntos outliers (rojos, destacados)
        if len(outliers_vals) > 0:
            # Muestra de outliers (máx 100)
            if len(outliers_vals) > 100:
                outliers_muestra = outliers_vals.sample(100, random_state=42)
            else:
                outliers_muestra = outliers_vals

            fig_strip.add_trace(go.Scatter(
                y=outliers_muestra,
                x=[var] * len(outliers_muestra),
                mode='markers',
                marker=dict(color='#E53E3E', size=5, opacity=0.7,
                            line=dict(width=0.5, color='white')),
                name='Outliers',
                showlegend=(i == 0),
                legendgroup='outliers',
                hovertemplate=f'{var}: %{{y:.2f}}<extra>Outlier</extra>'
            ), row=row, col=col_idx)

    fig_strip.update_layout(
        title='🔴 Stripplot: Outliers Reales por Variable<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Top variables con más outliers | ⬜ Normales | 🔴 Outliers (fuera de 1.5×IQR)</span>',
        height=350 * n_rows_s,
        margin=dict(t=90, b=60, l=60, r=40),
        showlegend=True
    )
    fig_strip.write_html(RUTA_GRAFICOS / 'm04_stripplot.html', include_plotlyjs='cdn')
    print(f'✅ Stripplot: {n_vars_s} variables con outliers')
else:
    print('✅ Sin variables con outliers — no se genera stripplot')


✅ Stripplot: 6 variables con outliers


In [11]:
# ============================================================================
# CELDA: CONSTRUIR SECCIONES HTML
# ============================================================================

# --- KPIs ---
KPIS = [
    {'valor': str(len(num_cols)), 'titulo': 'Variables Numéricas'},
    {'valor': str(len(continuas)), 'titulo': 'Continuas'},
    {'valor': str(len(discretas)), 'titulo': 'Discretas'},
    {'valor': str(n_con_outliers), 'titulo': 'Con Outliers'},
]
kpis_html = generar_kpis_html(KPIS)

# --- CSS modales ---
modal_css = '''
<style>
.modal-overlay {
    display: none; position: fixed; top: 0; left: 0;
    width: 100%; height: 100%;
    background: rgba(0,0,0,0.7); z-index: 1000;
    justify-content: center; align-items: center;
}
.modal-content {
    background: white; border-radius: 10px;
    width: 90%; max-width: 1000px; max-height: 90%; overflow: auto;
}
.modal-header {
    display: flex; justify-content: space-between; align-items: center;
    padding: 15px 20px; border-bottom: 1px solid #e2e8f0;
}
.modal-close { background: none; border: none; font-size: 1.5em; cursor: pointer; color: #718096; }
.modal-close:hover { color: #e53e3e; }
.btn-modal {
    padding: 12px 24px; background: #E53E3E; color: white;
    border: none; border-radius: 8px; font-weight: 600; cursor: pointer;
}
.btn-modal:hover { background: #C53030; }
</style>
'''

# --- Sección: Variables excluidas ---
seccion_info = generar_seccion_html(
    titulo="Variables Excluidas", icono="ℹ️",
    contenido='''<div style="background:#EBF8FF; border-left:4px solid #3182ce; padding:15px; color:#2B6CB0;">
        <strong>📌 Variables excluidas del análisis:</strong><br>
        • IDs: per_id_ficticio, exp_tit_id<br>
        • Fechas: curso_aca_ini, curso_aca, curso_aca_fin<br>
        • Categóricas codificadas: sexo<br>
        • Ordinal numérica excluida por NaN estructurales (12.6%): orden_preferencia
        — corresponde a alumnos sin preinscripción (traslados, mayores de 25, accesos directos)
    </div>'''
)

# --- Clasificación ---
seccion_clasif = generar_seccion_html(
    titulo="Clasificación por Naturaleza", icono="📊",
    contenido='<iframe src="graficos/m04_clasificacion.html" width="100%" height="420" frameborder="0"></iframe>'
)

# --- Boxplots + tabla outliers + modal ---
tabla_outliers_html = df_outliers.to_html(
    classes='tabla-estadisticas', index=False,
    float_format=lambda x: f'{x:,.2f}'
) if len(df_outliers) > 0 else '<p style="text-align:center; color:#38a169;">✅ Sin valores extremos detectados</p>'

contenido_boxplots = f'''
<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
    <iframe src="graficos/m04_boxplots.html" width="100%" height="440" frameborder="0"></iframe>
    <div>
        <div style="max-height:350px; overflow-y:auto; margin-bottom:15px;">{tabla_outliers_html}</div>
        <div style="text-align:center;">
            <button class="btn-modal" onclick="document.getElementById('modal-boxplots').style.display='flex'">
                👁️ Ver gráfico con datos originales
            </button>
        </div>
    </div>
</div>
<div id="modal-boxplots" class="modal-overlay" onclick="if(event.target==this)this.style.display='none'">
    <div class="modal-content">
        <div class="modal-header">
            <h3 style="margin:0;">📦 Distribuciones (con outliers)</h3>
            <button class="modal-close" onclick="document.getElementById('modal-boxplots').style.display='none'">×</button>
        </div>
        <iframe src="graficos/m04_boxplots_full.html" width="100%" height="500" frameborder="0"></iframe>
    </div>
</div>
'''
seccion_boxplots = generar_seccion_html(
    titulo="Distribuciones (sin outliers extremos)", icono="📦",
    contenido=contenido_boxplots
)

# --- Forma ---
tabla_forma_html = df_forma.to_html(classes='tabla-estadisticas', index=False,
    float_format=lambda x: f'{x:.2f}')
seccion_forma = generar_seccion_html(
    titulo="Forma de las Distribuciones (sin outliers)", icono="📐",
    contenido=f'''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m04_forma.html" width="100%" height="420" frameborder="0"></iframe>
        <div style="max-height:420px; overflow-y:auto;">{tabla_forma_html}</div>
    </div>'''
)

# --- Histogramas ---
altura_hist = 300 * n_rows_grid + 100
seccion_hist = generar_seccion_html(
    titulo="Distribución de Frecuencias", icono="📈",
    contenido=f'''<p style="color:#718096; margin-bottom:10px;">Leyenda: ── KDE | <span style="color:red;">|</span> Media | <span style="color:green;">┊</span> Mediana</p>
        <iframe src="graficos/m04_histogramas.html" width="100%" height="{altura_hist}" frameborder="0"></iframe>'''
)

# --- QQ-Plots ---
seccion_qq = generar_seccion_html(
    titulo="QQ-Plot: ¿Distribución Normal?", icono="📉",
    contenido=f'<iframe src="graficos/m04_qqplot.html" width="100%" height="{altura_hist}" frameborder="0"></iframe>'
)

# --- Violinplots ---
altura_violin = 350 * n_rows_v + 100 if vars_violin and curso_col else 300
seccion_violin = generar_seccion_html(
    titulo="Distribuciones por Curso (Violinplots)", icono="🎻",
    contenido=f'<iframe src="graficos/m04_violinplots.html" width="100%" height="{altura_violin}" frameborder="0"></iframe>'
    if vars_violin and curso_col else '<p style="text-align:center; color:#718096;">No disponible</p>'
)

# --- Stripplot ---
altura_strip = 350 * n_rows_s + 100 if vars_strip else 300
seccion_strip = generar_seccion_html(
    titulo="Outliers Reales (Stripplot)", icono="🔴",
    contenido=f'<iframe src="graficos/m04_stripplot.html" width="100%" height="{altura_strip}" frameborder="0"></iframe>'
    if vars_strip else '<p style="text-align:center; color:#38a169;">✅ Sin outliers significativos</p>'
)

# --- Tabla estadísticas ---
tabla_stats_html = df_stats.to_html(classes='tabla-estadisticas', index=False,
    float_format=lambda x: f'{x:.2f}')
seccion_tabla = generar_seccion_html(
    titulo="Estadísticas Descriptivas", icono="📋",
    contenido=f'<div style="overflow-x:auto; max-height:400px; overflow-y:auto;">{tabla_stats_html}</div>'
)

print('✅ Secciones HTML construidas')


✅ Secciones HTML construidas


In [12]:
# ============================================================================
# CELDA 11: GENERAR HTML — PÁGINA M04 CON MODALES
# ============================================================================
# Ensambla la página con:
# - KPIs: variables, continuas, discretas, con outliers
# - Info variables excluidas
# - Clasificación (donut)
# - Boxplots + tabla outliers + modal popup
# - Forma (asimetría + curtosis)
# - Histogramas con KDE
# - QQ-Plots
# - Tabla estadísticas descriptivas
# ============================================================================

print('=' * 60)
print('GENERANDO HTML')
print('=' * 60)

# --- Ensamblar contenido ---
contenido_html = (modal_css + kpis_html + seccion_info + seccion_clasif +
                  seccion_boxplots + seccion_forma + seccion_hist + seccion_qq +
                  seccion_violin + seccion_strip + seccion_tabla)

# --- Generar HTML ---
html_completo = render_pagina_desde_fichero(
    'f2_m04_univariante_num.ipynb',
    contenido_html,
    carpeta_notebook='fase2_eda'
)
ruta_html = RUTA_FASE2 / "m04_univariante_num.html"
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')
print(f"\n✅ HTML: {ruta_html}")

GENERANDO HTML
✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m04_univariante_num.html

✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m04_univariante_num.html


In [13]:
# ============================================================================
# CELDA 12: RESUMEN DE EJECUCIÓN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M04 COMPLETADO')
print('=' * 60)
print('📊 Clasificación por naturaleza')
print('📦 Boxplots sin outliers + Tabla + Modal popup')
print('📐 Asimetría + Curtosis + Tabla')
print('📈 Histogramas con KDE')
print('📉 QQ-Plots')
print('🎻 Violinplots por curso (NUEVO)')
print('🔴 Stripplot outliers (NUEVO)')
print('📋 Tabla estadísticas')
print('📌 Siguiente: f2_m05_univariante_cat.ipynb')


✅ F2-M04 COMPLETADO
📊 Clasificación por naturaleza
📦 Boxplots sin outliers + Tabla + Modal popup
📐 Asimetría + Curtosis + Tabla
📈 Histogramas con KDE
📉 QQ-Plots
🎻 Violinplots por curso (NUEVO)
🔴 Stripplot outliers (NUEVO)
📋 Tabla estadísticas
📌 Siguiente: f2_m05_univariante_cat.ipynb
